# Linear & Polynomial Regression — From `sklearn` to From-Scratch

A single, step-by-step teaching notebook covering:

1. Data preparation (encoding, scaling, train/test split)
2. Linear Regression with `sklearn` — univariate, then multivariate
3. Evaluating regression models (R², Adjusted R²)
4. Polynomial Regression with `sklearn`
5. Underfitting, Overfitting & the Bias–Variance tradeoff
6. Regularization — Ridge (L2), Lasso (L1) & ElasticNet
7. Hyperparameters & how to tune them (degree, regularization rate)
8. Cross-Validation (train / validation / test)
9. K-Fold Cross-Validation
10. Hand-coded Linear Regression from scratch (Gradient Descent)
11. Why feature scaling matters for Gradient Descent
12. Linear Regression with `statsmodels` (the "why" behind the numbers)
13. Checking the assumptions of Linear Regression (multicollinearity, residual normality, heteroskedasticity)

We use the **Cars24 used-car price dataset** throughout so every section builds on the same data.


## Part 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
np.random.seed(1)


### Load the data

We download the Cars24 cleaned dataset (car `make`, `model`, `age`, `km_driven`, etc. → target: `selling_price`).

In [ ]:
!gdown 1bwRmKkPwmLKiqOgQ_LnKH0Vsc3mJKmVR

df = pd.read_csv('cars24-car-price-cleaned.csv')
df.head()


In [ ]:
df.shape


## Part 1 — Data Preparation

### 1.1 Encoding categorical columns

`make` and `model` are text categories — a model can't use them directly. We'll use **mean target encoding**:
replace each category with the average `selling_price` for that category.

In [ ]:
df['model'].nunique(), df['make'].nunique()


In [ ]:
df['make'] = df.groupby('make')['selling_price'].transform('mean')
df['model'] = df.groupby('model')['selling_price'].transform('mean')
df.head()


### 1.2 Scaling the data

Notice the columns live on very different scales — e.g. `km_driven` is in the tens of thousands, while `age` is in the tens.
Gradient-based models (and distance-based ones) train more reliably when every feature is on a comparable scale, so we apply `MinMaxScaler`.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
df_scaled.head()


### 1.3 Train / test split

We hold out 30% of the data for testing so we can measure how well the model **generalizes** to unseen data.

In [ ]:
from sklearn.model_selection import train_test_split

y = df_scaled['selling_price']
X = df_scaled.drop('selling_price', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
X_train.shape, X_test.shape


## Part 2 — Linear Regression with `scikit-learn`

### 2.1 Univariate Linear Regression

Let's start with a single feature (`model`, the target-encoded average price for that model) to build intuition.

In [ ]:
from sklearn.linear_model import LinearRegression

X1_train = X_train[['model']]
X1_test = X_test[['model']]

uni_model = LinearRegression()
uni_model.fit(X1_train, y_train)

uni_model.coef_, uni_model.intercept_


In [ ]:
y_hat_uni = uni_model.predict(X[['model']])

plt.figure(figsize=(6, 4))
plt.scatter(X[['model']], y, label='data', alpha=0.5)
plt.scatter(X[['model']], y_hat_uni, color='orange', label='prediction', alpha=0.5)
plt.xlabel('model (mean-encoded)')
plt.ylabel('selling_price (scaled)')
plt.legend()
plt.show()


The predictions line up along a straight line — exactly what we'd expect from fitting `y = w*x + b` on a single feature.

### 2.2 Multivariate Linear Regression

Now let's use **all** the input features at once.

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

model.coef_, model.intercept_


In [ ]:
y_hat = model.predict(X_test)

plt.figure(figsize=(5, 5))
plt.scatter(y_test, y_hat, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='perfect prediction')
plt.xlabel('actual selling_price')
plt.ylabel('predicted selling_price')
plt.legend()
plt.show()


## Part 3 — Evaluating Regression Models

### R² (coefficient of determination)
$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

It compares model error to the error of a "predict the mean" baseline. 1.0 is a perfect fit, 0 means no better than the mean.

### Adjusted R²

Plain R² never decreases when you add more features, even useless ones. Adjusted R² penalizes extra features that don't earn their keep:
$$\bar{R}^2 = 1 - \frac{(1-R^2)(n-1)}{n-p-1}$$
where $n$ = number of samples, $p$ = number of features.

In [ ]:
def r2_score(y, y_pred):
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    return 1 - ss_res / ss_tot

def adj_r2_score(r_sq, X, y):
    n, p = X.shape[0], X.shape[1]
    return 1 - ((1 - r_sq) * (n - 1)) / (n - p - 1)


In [ ]:
# sklearn's built-in .score() already computes R^2
print('Train R2:', model.score(X_train, y_train))
print('Test  R2:', model.score(X_test, y_test))


In [ ]:
test_r2 = r2_score(y_test.values, model.predict(X_test))
print('Test R2 (manual):', test_r2)
print('Test Adjusted R2:', adj_r2_score(test_r2, X_test, y_test))


## Part 4 — Polynomial Regression with `scikit-learn`

Linear regression assumes a straight-line (hyperplane) relationship. Let's see what happens when the true relationship is curved.

In [ ]:
np.random.seed(1)
X_poly_demo = np.random.rand(50, 1)
y_poly_demo = (
    0.7 * X_poly_demo ** 5
    - 2.1 * X_poly_demo ** 4
    + 2.7 * X_poly_demo ** 3
    + 3.5 * X_poly_demo ** 2
    + 0.3 * X_poly_demo
    + 0.4 * np.random.rand(50, 1)  # noise
)

plt.figure(figsize=(5, 3))
plt.scatter(X_poly_demo, y_poly_demo)
plt.xlabel('X')
plt.ylabel('y')
plt.title('A non-linear dataset')
plt.show()


### 4.1 Fitting a plain (degree-1) Linear Regression

In [ ]:
lin_demo = LinearRegression()
lin_demo.fit(X_poly_demo, y_poly_demo)
pred_demo = lin_demo.predict(X_poly_demo)

print('R2:', r2_score(y_poly_demo, pred_demo))

plt.figure(figsize=(5, 3))
plt.scatter(X_poly_demo, y_poly_demo, label='samples')
plt.plot(X_poly_demo, pred_demo, label='prediction', color='orange')
plt.legend()
plt.show()


A straight line clearly can't capture this curve — the model **underfits**.

### 4.2 Adding a polynomial feature ($x^2$)

The trick: engineer a new feature $f_2 = f_1^2$ and fit a *linear* model on $[f_1, f_2]$.
$$\hat{y} = w_0 + w_1 f_1 + w_2 f_1^2$$
This is still "linear regression" in the sense that it's linear in the **weights** — only the *feature* is non-linear.

In [ ]:
X_deg2 = np.hstack([X_poly_demo, X_poly_demo ** 2])

model_deg2 = LinearRegression()
model_deg2.fit(X_deg2, y_poly_demo)
pred_deg2 = model_deg2.predict(X_deg2)

r2_deg2 = r2_score(y_poly_demo, pred_deg2)
print('Adj. R2 (degree 2):', adj_r2_score(r2_deg2, X_poly_demo, y_poly_demo))

plt.figure(figsize=(5, 3))
plt.scatter(X_poly_demo, y_poly_demo, label='samples')
plt.scatter(X_poly_demo, pred_deg2, color='orange', label='prediction')
plt.legend()
plt.show()


### 4.3 Sweeping degrees 1–6 with `PolynomialFeatures`

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.ravel()

for d in range(1, 7):
    poly = PolynomialFeatures(degree=d, include_bias=False)
    X_d = poly.fit_transform(X_poly_demo)

    m = LinearRegression()
    m.fit(X_d, y_poly_demo)
    pred_d = m.predict(X_d)
    r2_d = r2_score(y_poly_demo, pred_d)
    adj_d = adj_r2_score(r2_d, X_poly_demo, y_poly_demo)

    order = np.argsort(X_poly_demo[:, 0])
    axes[d - 1].scatter(X_poly_demo, y_poly_demo, s=15)
    axes[d - 1].plot(X_poly_demo[order], pred_d[order], color='orange')
    axes[d - 1].set_title(f'degree={d}, adj R2={adj_d:.3f}')

plt.tight_layout()
plt.show()


## Part 5 — Underfitting, Overfitting & Bias–Variance

Higher-degree polynomials fit the *training* curve better and better — but does that mean degree 40 is a good model? Let's check train **and test** performance as degree grows.

In [ ]:
np.random.seed(1)
X_bv = np.random.rand(100, 1)
y_bv = (
    0.7 * X_bv ** 5 - 2.1 * X_bv ** 4 + 2.7 * X_bv ** 3
    + 3.5 * X_bv ** 2 + 0.3 * X_bv + 0.4 * np.random.rand(100, 1)
)

Xb_train, Xb_test, yb_train, yb_test = train_test_split(X_bv, y_bv, test_size=0.3, random_state=1)

train_scores, test_scores = [], []
degrees = range(1, 30)

for d in degrees:
    poly = PolynomialFeatures(degree=d, include_bias=False)
    Xb_train_d = poly.fit_transform(Xb_train)
    Xb_test_d = poly.transform(Xb_test)

    m = LinearRegression()
    m.fit(Xb_train_d, yb_train)

    train_r2 = r2_score(yb_train, m.predict(Xb_train_d))
    test_r2 = r2_score(yb_test, m.predict(Xb_test_d))
    train_scores.append(adj_r2_score(train_r2, Xb_train_d, yb_train))
    test_scores.append(adj_r2_score(test_r2, Xb_test_d, yb_test))


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(list(degrees), train_scores, label='train')
plt.plot(list(degrees), test_scores, label='test')
plt.xlabel('polynomial degree')
plt.ylabel('Adjusted R2')
plt.legend()
plt.title('Train vs Test performance as model complexity grows')
plt.show()


**Reading the plot:**
- **Low degree (left):** both train and test scores are poor → **underfitting** (high bias) — the model is too simple to capture the pattern.
- **High degree (right):** train score stays high but test score collapses → **overfitting** (high variance) — the model has started memorizing noise instead of the underlying pattern.
- **The sweet spot** is where test performance peaks — usually a fairly low degree (here, around 2–5). This is **Occam's Razor**: prefer the simplest model that explains the data well.

**Bias–variance tradeoff, intuitively:**
- *Bias* = error from a model being too simple to capture the true pattern (systematically "off-target").
- *Variance* = error from a model being too sensitive to the specific training sample (predictions swing wildly with small data changes).
- The ideal model has **low bias and low variance** — good fit *and* consistent predictions on new data.

## Part 6 — Regularization: Ridge, Lasso & ElasticNet

We just saw that high-degree polynomials **overfit** — they drive training error down by growing large, wiggly weights that chase noise. Regularization fights this directly by **penalizing large weights** in the loss function, instead of only penalizing prediction error.

$$\text{Loss} = \text{MSE}(y, \hat{y}) \;+\; \lambda \times \text{penalty}(W)$$

- $\lambda$ (often called `alpha` in `sklearn`) controls the *strength* of the penalty.
- Too small $\lambda$ → penalty barely matters → still overfits.
- Too large $\lambda$ → weights get pushed too close to 0 → the model underfits.
- The right $\lambda$ finds the sweet spot: enough freedom for MSE to fit real signal, enough pressure from the penalty to ignore noise.

### L2 penalty — Ridge Regression
$$\text{penalty}(W) = \sum_{j=1}^{d} w_j^2$$
Shrinks all weights smoothly toward (but rarely exactly to) zero.

### L1 penalty — Lasso Regression
$$\text{penalty}(W) = \sum_{j=1}^{d} |w_j|$$
Can push weights **exactly to zero** — effectively performing feature selection. This happens because the derivative of $|w_j|$ is a constant ($\pm 1$) regardless of how small $w_j$ already is, while the derivative of $w_j^2$ (which is $2w_j$) shrinks as $w_j$ approaches zero — so L2's "pull" toward zero weakens near zero, while L1's stays constant and can zero a weight out entirely. This makes L1 useful when you suspect many features are irrelevant, and L2 useful when you want to keep all features but shrink their influence.

### Elastic Net — combining both
$$\text{penalty}(W) = r \sum_{j=1}^{d} |w_j| \;+\; (1-r) \sum_{j=1}^{d} w_j^2$$
A mix of L1 and L2, useful when you're not sure which to prefer — `l1_ratio` (our $r$ above) controls the blend.

### 6.1 Regularization in action: fixing an overfit polynomial model

Let's push our polynomial regression from Part 5 to a deliberately overfit degree, then see how Ridge and Lasso rescue it. We standardize the polynomial features first — regularization penalizes weight *magnitude*, so features must be on comparable scales for the penalty to be fair.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

overfit_degree = 25
poly_of = PolynomialFeatures(degree=overfit_degree, include_bias=False)
Xb_train_of = poly_of.fit_transform(Xb_train)
Xb_test_of = poly_of.transform(Xb_test)

poly_scaler = StandardScaler()
Xb_train_of_scaled = poly_scaler.fit_transform(Xb_train_of)
Xb_test_of_scaled = poly_scaler.transform(Xb_test_of)

plain_of = LinearRegression()
plain_of.fit(Xb_train_of_scaled, yb_train)

print('Plain Linear Regression, degree =', overfit_degree)
print('  Train MSE:', mean_squared_error(yb_train, plain_of.predict(Xb_train_of_scaled)))
print('  Test  MSE:', mean_squared_error(yb_test, plain_of.predict(Xb_test_of_scaled)))


Train MSE is tiny while test MSE is much higher — a textbook overfit. Now let's regularize.

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

ridge_of = Ridge(alpha=1.0)
lasso_of = Lasso(alpha=0.01)
elastic_of = ElasticNet(alpha=0.01, l1_ratio=0.5)

for name, m in [('Ridge', ridge_of), ('Lasso', lasso_of), ('ElasticNet', elastic_of)]:
    m.fit(Xb_train_of_scaled, yb_train)
    train_mse = mean_squared_error(yb_train, m.predict(Xb_train_of_scaled))
    test_mse = mean_squared_error(yb_test, m.predict(Xb_test_of_scaled))
    print(f'{name:10s}  train MSE={train_mse:.5f}   test MSE={test_mse:.5f}')


All three regularized models bring the test MSE much closer to (or below) the plain model's — the penalty term stopped the model from chasing noise in the training data.

### 6.2 L1 vs L2: sparsity in practice

Let's check how many weights Lasso actually zeroes out compared to Ridge, on the same overfit features.

In [ ]:
n_zero_lasso = np.sum(np.isclose(lasso_of.coef_, 0))
n_zero_ridge = np.sum(np.isclose(ridge_of.coef_, 0))

print(f'Lasso: {n_zero_lasso} / {len(lasso_of.coef_)} weights driven to (near) zero')
print(f'Ridge: {n_zero_ridge} / {len(ridge_of.coef_)} weights driven to (near) zero')


Lasso zeroes out a meaningful chunk of the high-degree polynomial terms — effectively deciding those terms aren't needed. Ridge shrinks everything a little, but rarely reaches exactly zero.

## Part 7 — Hyperparameters & Tuning Them

### Parameters vs. Hyperparameters

- **Parameters** ($W$, $b$) are *learned* by the model during training (e.g., via gradient descent).
- **Hyperparameters** are *set by us* before training — things like the polynomial `degree` or the regularization strength `alpha` ($\lambda$). The model can't learn these on its own; we have to search for good values.

### How do we choose hyperparameter values?

A natural approach: try a range of values, train a model for each, and see which value gives the best performance **on data the model wasn't trained on**. Let's do exactly that for `degree` and then for Ridge's `alpha`, plotting train vs. test performance to visualize the tradeoff, just like we did for Part 5's degree sweep.

In [ ]:
def adj_r2(X, y, r2):
    return 1 - ((1 - r2) * (len(y) - 1)) / (len(y) - X.shape[1] - 1)


We'll use a fresh, larger synthetic dataset (1000 points) so the tuning curves are smoother.

In [ ]:
from sklearn.pipeline import make_pipeline

np.random.seed(2)
X_ht = np.random.rand(1000, 1)
y_ht = (
    0.7 * X_ht ** 5 - 2.1 * X_ht ** 4 + 2.3 * X_ht ** 3
    + 0.2 * X_ht ** 2 + 0.3 * X_ht + 0.4 * np.random.rand(1000, 1)
)

Xht_train, Xht_test, yht_train, yht_test = train_test_split(X_ht, y_ht, test_size=0.2, random_state=1)
Xht_train.shape, Xht_test.shape


### 7.1 Tuning `degree` (with Ridge as the base model)

In [ ]:
max_degree = 25
train_scores, test_scores = [], []
scaler_ht = StandardScaler()

for degree in range(1, max_degree):
    pipe = make_pipeline(PolynomialFeatures(degree), scaler_ht, Ridge())
    pipe.fit(Xht_train, yht_train)
    train_scores.append(adj_r2(Xht_train, yht_train, pipe.score(Xht_train, yht_train)))
    test_scores.append(adj_r2(Xht_test, yht_test, pipe.score(Xht_test, yht_test)))

plt.figure(figsize=(8, 4))
plt.plot(range(1, max_degree), train_scores, label='train')
plt.plot(range(1, max_degree), test_scores, label='test')
plt.xlabel('degree')
plt.ylabel('Adj. R2')
plt.legend()
plt.grid(True)
plt.show()


The test curve peaks early (around degree ≈ 3) and then flattens/declines — that's our chosen `degree`.

### 7.2 Tuning `alpha` (regularization rate), with `degree` fixed at 3

In [ ]:
train_scores, test_scores = [], []
rate_list = [0.01, 0.1, 1, 5, 10]

for rate in rate_list:
    pipe = make_pipeline(PolynomialFeatures(3), scaler_ht, Ridge(alpha=rate))
    pipe.fit(Xht_train, yht_train)
    train_scores.append(adj_r2(Xht_train, yht_train, pipe.score(Xht_train, yht_train)))
    test_scores.append(adj_r2(Xht_test, yht_test, pipe.score(Xht_test, yht_test)))

plt.figure(figsize=(6, 4))
plt.plot(rate_list, train_scores, label='train')
plt.plot(rate_list, test_scores, label='test')
plt.xlabel('lambda (alpha)')
plt.ylabel('Adj. R2')
plt.legend()
plt.grid(True)
plt.show()

test_scores


The smallest `alpha` tested performs best here — with `degree=3` we're already close to the right complexity, so a light regularization touch is enough.

## Part 8 — Cross-Validation

There's a subtle problem with what we just did: we tuned `degree` and `alpha` by repeatedly checking performance on the **same test set**. That means the test set has quietly influenced our choices — it's no longer a fair, untouched estimate of real-world performance.

**The fix:** split the data into **three** parts:
- **Train** — fit the model's parameters.
- **Validation** — compare hyperparameter choices against each other.
- **Test** — touch this exactly once, at the very end, to report the final, honest performance.

In [ ]:
# 60% train, 20% validation, 20% test
X_tr_cv, Xcv_test, y_tr_cv, ycv_test = train_test_split(X_ht, y_ht, test_size=0.2, random_state=1)
Xcv_train, Xcv_val, ycv_train, ycv_val = train_test_split(X_tr_cv, y_tr_cv, test_size=0.25, random_state=1)

Xcv_train.shape, Xcv_val.shape, Xcv_test.shape


**Step 1 — pick `degree` using train vs. validation:**

In [ ]:
max_degree = 25
train_scores, val_scores = [], []
scaler_cv = StandardScaler()

for degree in range(1, max_degree):
    pipe = make_pipeline(PolynomialFeatures(degree), scaler_cv, Ridge())
    pipe.fit(Xcv_train, ycv_train)
    train_scores.append(adj_r2(Xcv_train, ycv_train, pipe.score(Xcv_train, ycv_train)))
    val_scores.append(adj_r2(Xcv_val, ycv_val, pipe.score(Xcv_val, ycv_val)))

plt.figure(figsize=(8, 4))
plt.plot(range(1, max_degree), train_scores, label='train')
plt.plot(range(1, max_degree), val_scores, label='validation')
plt.xlabel('degree')
plt.ylabel('Adj. R2')
plt.legend()
plt.grid(True)
plt.show()


**Step 2 — with `degree=3` fixed, pick `alpha` using train vs. validation:**

In [ ]:
train_scores, val_scores = [], []
rate_list = [0.01, 0.1, 1, 5, 10]

for rate in rate_list:
    pipe = make_pipeline(PolynomialFeatures(3), scaler_cv, Ridge(alpha=rate))
    pipe.fit(Xcv_train, ycv_train)
    train_scores.append(adj_r2(Xcv_train, ycv_train, pipe.score(Xcv_train, ycv_train)))
    val_scores.append(adj_r2(Xcv_val, ycv_val, pipe.score(Xcv_val, ycv_val)))

plt.figure(figsize=(6, 4))
plt.plot(rate_list, train_scores, label='train')
plt.plot(rate_list, val_scores, label='validation')
plt.xlabel('lambda (alpha)')
plt.ylabel('Adj. R2')
plt.legend()
plt.grid(True)
plt.show()


**Step 3 — now, and only now, evaluate on the untouched test set:**

In [ ]:
final_pipe = make_pipeline(PolynomialFeatures(3), scaler_cv, Ridge(alpha=0.01))
final_pipe.fit(Xcv_train, ycv_train)

print('Training Score:  ', adj_r2(Xcv_train, ycv_train, final_pipe.score(Xcv_train, ycv_train)))
print('Validation Score:', adj_r2(Xcv_val, ycv_val, final_pipe.score(Xcv_val, ycv_val)))
print('Test Score:      ', adj_r2(Xcv_test, ycv_test, final_pipe.score(Xcv_test, ycv_test)))


The test score is a bit lower than train/validation — expected, since the model never got to adjust to this data at all. This gap is the most honest estimate of how the model will do on brand-new, unseen data.

**Try it yourself:** swap `Ridge` for `Lasso` or `ElasticNet` above and see how the tuning curves and final test score change.

## Part 9 — K-Fold Cross-Validation

A single train/validation/test split has a downside: with a **small dataset**, a lot of data gets "used up" just sitting in the validation and test sets instead of helping the model learn — and results can be noisy depending on which points happened to land in which split.

**K-Fold Cross-Validation** fixes this for small datasets: split the data into $K$ equal folds; for each hyperparameter choice, train $K$ times, each time holding out a *different* fold as the validation set, then average the scores. Every data point gets used for training in $K-1$ of the $K$ runs — much more efficient use of limited data (at the cost of $K\times$ the training time).

In [ ]:
# A small dataset, where K-Fold CV is most useful
np.random.seed(2)
X_kf = np.random.rand(100, 1)
y_kf = (
    0.7 * X_kf ** 5 - 2.1 * X_kf ** 4 + 2.3 * X_kf ** 3
    + 0.2 * X_kf ** 2 + 0.3 * X_kf + 0.4 * np.random.rand(100, 1)
)


In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=10)


In [ ]:
max_degree = 15
train_scores, val_scores = [], []
scaler_kf = StandardScaler()

for degree in range(1, max_degree):
    fold_train, fold_val = [], []
    for train_idx, val_idx in kf.split(X_kf):
        X_tr_fold, X_val_fold = X_kf[train_idx], X_kf[val_idx]
        y_tr_fold, y_val_fold = y_kf[train_idx], y_kf[val_idx]

        pipe = make_pipeline(PolynomialFeatures(degree), scaler_kf, LinearRegression())
        pipe.fit(X_tr_fold, y_tr_fold)

        fold_train.append(pipe.score(X_tr_fold, y_tr_fold))
        fold_val.append(pipe.score(X_val_fold, y_val_fold))

    train_scores.append(np.mean(fold_train))
    val_scores.append(np.mean(fold_val))


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(range(1, max_degree), train_scores, label='train')
plt.plot(range(1, max_degree), val_scores, label='validation')
plt.xlabel('degree')
plt.ylabel('Average R2 score')
plt.legend()
plt.grid(True)
plt.show()


**Note:**
- K-Fold CV can be computationally expensive (it trains $K$ models per hyperparameter value instead of 1).
- Because of that cost, it's mainly worth it for **small datasets**, where a single train/val split would waste too much data or give a noisy estimate.
- For large datasets, a plain train/validation/test split (Part 8) usually already gives a stable, reliable estimate — full K-Fold CV isn't necessary.

## Part 10 — Hand-Coded Linear Regression (From Scratch)

`sklearn`'s `LinearRegression` solves for the weights internally. Let's build the same thing ourselves using **Gradient Descent**, to see exactly what's happening under the hood.

**The idea:**
1. Start with weights `W` (and bias `b`) at zero.
2. Compute predictions: $\hat{y} = XW + b$
3. Compute the loss (Mean Squared Error) between $\hat{y}$ and the true $y$.
4. Compute the gradient of the loss with respect to `W` and `b`.
5. Nudge `W` and `b` a small step (`learning_rate`) in the direction that reduces the loss.
6. Repeat for a number of `iterations`.

We'll build this step by step as a Python class.

### Step 1 — the constructor

In [ ]:
class LinearRegressionScratch:
    def __init__(self, learning_rate=0.01, iterations=5):
        self.learning_rate = learning_rate
        self.iterations = iterations


### Step 2 — `predict`: $\hat{y} = XW + b$

In [ ]:
def predict(self, X):
    return np.dot(X, self.W) + self.b

LinearRegressionScratch.predict = predict


### Step 3 — `score`: R² evaluation metric

In [ ]:
def r2_score_method(self, X, y):
    y_pred = self.predict(X)
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot

LinearRegressionScratch.score = r2_score_method


### Step 4 — `update_weights`: one gradient descent step

For MSE loss, the gradients work out to:
$$dW = -\frac{2}{m} X^T (y - \hat{y}) \qquad db = -\frac{2}{m} \sum (y - \hat{y})$$

Using **all** `m` training points for a single update is called **Batch Gradient Descent**. (Using 1 point at a time is Stochastic GD; a small chunk at a time is Mini-Batch GD.)

In [ ]:
def update_weights(self):
    y_pred = self.predict(self.X)

    dW = -(2 * self.X.T.dot(self.Y - y_pred)) / self.m
    db = -2 * np.sum(self.Y - y_pred) / self.m

    self.W = self.W - self.learning_rate * dW
    self.b = self.b - self.learning_rate * db

    # track loss for plotting later
    loss = np.mean((self.Y - y_pred) ** 2)
    self.error_list.append(loss)
    return self

LinearRegressionScratch.update_weights = update_weights


### Step 5 — `fit`: initialize weights and run gradient descent for `iterations` steps

In [ ]:
def fit(self, X, Y):
    self.m, self.d = X.shape  # no. of samples, no. of features
    self.W = np.zeros(self.d)
    self.b = 0
    self.X = np.asarray(X)
    self.Y = np.asarray(Y)
    self.error_list = []

    for _ in range(self.iterations):
        self.update_weights()
    return self

LinearRegressionScratch.fit = fit


### Step 6 — train it on the Cars24 data and compare with `sklearn`

In [ ]:
scratch_model = LinearRegressionScratch(learning_rate=0.1, iterations=200)
scratch_model.fit(X_train, y_train)

print('Train R2:', scratch_model.score(X_train, np.asarray(y_train)))
print('Test  R2:', scratch_model.score(X_test, np.asarray(y_test)))


In [ ]:
print('sklearn  coef:', model.coef_[:5])
print('scratch  coef:', scratch_model.W[:5])
print('sklearn  intercept:', model.intercept_)
print('scratch  intercept:', scratch_model.b)


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(scratch_model.error_list)
plt.xlabel('iteration')
plt.ylabel('MSE loss')
plt.title('Loss reduction over training')
plt.show()


With enough iterations and a well-tuned learning rate, the from-scratch weights converge close to `sklearn`'s closed-form solution.

## Part 11 — Why Feature Scaling Matters for Gradient Descent

We already scaled our data with `MinMaxScaler` in Part 1. Let's see what happens to gradient descent if we **hadn't**.

When features live on very different scales, the loss surface becomes elongated/skewed. Gradient descent then zig-zags and converges much more slowly (or needs a tiny learning rate to avoid diverging).

In [ ]:
# Un-scaled version of the same features, for comparison
y_raw = df['selling_price']
X_raw = df.drop('selling_price', axis=1)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_raw, y_raw, test_size=0.3, random_state=1)

unscaled_model = LinearRegressionScratch(learning_rate=0.1, iterations=200)
unscaled_model.fit(Xr_train, yr_train)

plt.figure(figsize=(6, 4))
plt.plot(scratch_model.error_list, label='scaled features')
plt.plot(unscaled_model.error_list, label='unscaled features')
plt.yscale('log')
plt.xlabel('iteration')
plt.ylabel('MSE loss (log scale)')
plt.legend()
plt.title('Scaling makes gradient descent converge far more reliably')
plt.show()


## Part 12 — Linear Regression with `statsmodels`

`sklearn` is built for prediction. `statsmodels` gives us the **statistical** view — p-values, confidence intervals, and diagnostics — which is what we need to check the *assumptions* of linear regression.

In [ ]:
import statsmodels.api as sm

# statsmodels does not add an intercept by default — we add it explicitly
X_train_sm = sm.add_constant(X_train)

ols_model = sm.OLS(y_train, X_train_sm)
ols_results = ols_model.fit()
print(ols_results.summary())


A few useful rows in that summary:
- **R-squared / Adj. R-squared** — same meaning as before.
- **coef** — the fitted weight for each feature (matches `sklearn`'s `.coef_`).
- **P>|t|** — whether a feature's effect is statistically significant (small p-value ⇒ significant).

We can confirm the predictions match `sklearn`:

In [ ]:
(ols_results.predict(X_train_sm).values[:5], model.predict(X_train)[:5])


## Part 13 — Checking the Assumptions of Linear Regression

Linear Regression is provably the best *linear unbiased* estimator when several statistical assumptions hold:
1. The relationship is (approximately) linear.
2. Features are not strongly multi-collinear.
3. Residuals (errors) are normally distributed.
4. Residuals have constant variance (**homoskedasticity**).
5. No autocorrelation between residuals (mainly relevant for time-series data).

Let's check a few of these on the Cars24 model.

### 13.1 Multi-collinearity — Variance Inflation Factor (VIF)

If two input features are highly correlated with each other, the model can't tell which one is "responsible" for the effect on `y`. VIF quantifies this for each feature — a common rule of thumb is to worry when **VIF > 5**.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_train_scaled_df = pd.DataFrame(X_train.values, columns=X_train.columns)

vif = pd.DataFrame()
vif['Feature'] = X_train_scaled_df.columns
vif['VIF'] = [variance_inflation_factor(X_train_scaled_df.values, i)
              for i in range(X_train_scaled_df.shape[1])]
vif = vif.sort_values('VIF', ascending=False).reset_index(drop=True)
vif


In [ ]:
# Iteratively drop the feature with the highest VIF until all are below the threshold
vif_threshold = 5
remaining_cols = list(X_train_scaled_df.columns)

while True:
    vif_iter = pd.DataFrame()
    vif_iter['Feature'] = remaining_cols
    vif_iter['VIF'] = [variance_inflation_factor(X_train_scaled_df[remaining_cols].values, i)
                        for i in range(len(remaining_cols))]
    max_vif = vif_iter['VIF'].max()
    if max_vif < vif_threshold or len(remaining_cols) <= 1:
        break
    worst_feature = vif_iter.sort_values('VIF', ascending=False).iloc[0]['Feature']
    remaining_cols.remove(worst_feature)

print('Features kept after VIF filtering:', remaining_cols)
vif_iter.sort_values('VIF', ascending=False)


### 13.2 Normality of residuals

We plot the residual distribution and run a **Shapiro-Wilk** normality test (statistic close to 1 ⇒ closer to normal).

In [ ]:
residuals = ols_results.predict(X_train_sm) - y_train

plt.figure(figsize=(6, 4))
sns.histplot(residuals, kde=True)
plt.xlabel('Residual')
plt.title('Distribution of residuals')
plt.show()


In [ ]:
from scipy import stats

shapiro_result = stats.shapiro(residuals)
print('Shapiro-Wilk statistic:', shapiro_result.statistic, ' p-value:', shapiro_result.pvalue)


### 13.3 Homoskedasticity — constant error variance

We want the spread of residuals to stay roughly constant as predicted values change. If the spread fans out (a "cone" shape), that's **heteroskedasticity**, and it violates the assumption.

In [ ]:
fitted_vals = ols_results.predict(X_train_sm)

plt.figure(figsize=(6, 4))
sns.scatterplot(x=fitted_vals, y=residuals, alpha=0.5)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted selling_price')
plt.ylabel('Residual')
plt.title('Residuals vs Predicted values')
plt.show()


In [ ]:
# Goldfeld-Quandt test: formally tests for heteroskedasticity
import statsmodels.stats.api as sms

gq_stat, gq_pvalue, _ = sms.het_goldfeldquandt(y_train, X_train_sm)
print(f'F statistic: {gq_stat:.3f},  p-value: {gq_pvalue:.3f}')


A high p-value (> 0.05) means we **fail to reject** the null hypothesis of equal variances — i.e., no strong evidence of heteroskedasticity. If heteroskedasticity *is* found, common fixes are a log/sqrt/Box-Cox transform of the target, or removing outliers.

## Summary

| Section | What we learned |
|---|---|
| `sklearn` Linear Regression | Fast, production-ready fitting — univariate & multivariate |
| R2 / Adjusted R2 | How to measure fit, and penalize unnecessary complexity |
| Polynomial Regression | Engineer non-linear features, still fit with linear regression |
| Underfit/Overfit | Model complexity vs. generalization — the bias-variance tradeoff |
| Regularization (Ridge/Lasso/ElasticNet) | Penalize large weights to curb overfitting; L1 also gives sparsity |
| Hyperparameter tuning | Sweeping degree/alpha against held-out performance |
| Cross-Validation | Train/validation/test split so the test set stays untouched until the end |
| K-Fold Cross-Validation | Reuse limited data efficiently by rotating the validation fold |
| From-scratch Linear Regression | What `sklearn` does internally, via Gradient Descent |
| Feature scaling | Why it matters for gradient-based optimization |
| `statsmodels` OLS | Statistical significance, not just prediction |
| Assumption checks | VIF (multicollinearity), residual normality, homoskedasticity |

Everything here uses the same Cars24 dataset (plus small synthetic datasets for the polynomial/regularization demos) end-to-end, so you can trace one problem through every technique.
